# Multi-Region API Management Deployment

This notebook deploys an Azure API Management service across two regions using Premium SKU.

In [ ]:
import os
import sys
import json
import logging
from datetime import datetime

notebook_dir = "/home/ubuntu/repos/TestDevin/multi-region-apim/notebooks"
sys.path.append(os.path.join(notebook_dir, "scripts"))
import utils

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

## Initialize Variables

In [ ]:
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group_name = "rg-multi-region-apim"
primary_location = "eastus"
secondary_location = "westeurope"
apim_name = f"apim-{os.getenv("AZURE_SUBSCRIPTION_ID", "")[:8]}"
apim_sku = "Premium"

## Verify Azure CLI Authentication

In [ ]:
def verify_azure_cli():
    logger.info("Verifying Azure CLI authentication...")
    result = utils.run_az_command("az account show")
    if not result:
        raise Exception("Azure CLI authentication failed")
    logger.info(f"Authenticated as: {result.get("user", {}).get("name")}")
    return result

verify_azure_cli()

## Create Resource Group

In [ ]:
def create_resource_group():
    logger.info(f"Creating resource group: {resource_group_name} in {primary_location}")
    command = f"az group create --name {resource_group_name} --location {primary_location}"
    result = utils.run_az_command(command)
    if not result:
        raise Exception("Resource group creation failed")
    logger.info("Resource group created successfully")
    return result

create_resource_group()

## Validate and Deploy Bicep Template

In [ ]:
def deploy_bicep():
    logger.info("Deploying Bicep template...")
    deployment_name = f"apim-deploy-{datetime.now().strftime("%Y%m%d-%H%M%S")}"
    
    # First, validate the template
    logger.info("Validating Bicep template...")
    validate_command = f"az deployment group validate --name {deployment_name} --resource-group {resource_group_name} --template-file ../bicep/main.bicep --parameters location={primary_location} secondaryLocation={secondary_location} apimName={apim_name}"
    validation_result = utils.run_az_command(validate_command)
    if not validation_result:
        raise Exception("Bicep template validation failed")
    logger.info("Template validation successful")
    
    # Then deploy if validation passes
    logger.info("Starting deployment...")
    command = f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file ../bicep/main.bicep --parameters location={primary_location} secondaryLocation={secondary_location} apimName={apim_name}"
    result = utils.run_az_command(command)
    if not result:
        raise Exception("Bicep deployment failed")
    return result

deploy_bicep()

## Get Deployment Outputs

In [ ]:
def get_deployment_outputs():
    logger.info("Getting deployment outputs...")
    command = f"az apim show -g {resource_group_name} -n {apim_name}"
    result = utils.run_az_command(command)
    if result:
        logger.info(f"Primary Gateway URL: {result.get("gatewayUrl")}")
        logger.info(f"Additional Locations: {result.get("additionalLocations", [])}")
        logger.info(f"Mock Backend API URL: {result.get("gatewayUrl")}/mock")
    return result

get_deployment_outputs()

## Verify Traffic Manager Configuration

In [ ]:
def verify_traffic_manager():
    logger.info("Verifying Traffic Manager deployment...")
    tm_name = f"{apim_name}-tm"
    command = f"az network traffic-manager profile show -g {resource_group_name} -n {tm_name}"
    result = utils.run_az_command(command)
    if result:
        logger.info(f"Traffic Manager URL: https://{result.get("dnsConfig", {}).get("relativeName")}.trafficmanager.net")
        
        # Check endpoint health
        endpoints_command = f"az network traffic-manager endpoint list -g {resource_group_name} --profile-name {tm_name}"
        endpoints = utils.run_az_command(endpoints_command)
        if endpoints:
            for endpoint in endpoints:
                logger.info(f"Endpoint {endpoint.get("name")}: {endpoint.get("properties", {}).get("endpointMonitorStatus", "Unknown")}")
    return result

verify_traffic_manager()